In [ ]:
from datasets import load_dataset
import pandas as pd
import subprocess
import json
import re
from datetime import datetime


In [ ]:
dataset = load_dataset("cais/mmlu", "econometrics")
test_data = dataset["test"]

print("Number of questions:", len(test_data))
test_data[0]


Number of questions: 114


{'question': 'Which one of the following is the most appropriate definition of a 99% confidence interval?',
 'subject': 'econometrics',
 'choices': ['99% of the time in repeated samples, the interval would contain the true value of the parameter',
  '99% of the time in repeated samples, the interval would contain the estimated value of the parameter',
  '99% of the time in repeated samples, the null hypothesis will be rejected',
  '99% of the time in repeated samples, the null hypothesis will not be rejected when it was false'],
 'answer': 0}

In [ ]:
models = [
    "llama3.1:8b",
    #"phi3:medium",
    "qwen2.5:7b",
    "qwen2.5:14b",
    "mistral-nemo:12b"
]

models


['llama3.1:8b', 'qwen2.5:7b', 'qwen2.5:14b', 'mistral-nemo:12b']

In [ ]:
def build_prompt(question, choices):
    A, B, C, D = choices
    return f"""
Answer the following econometrics question.

Respond ONLY in this JSON format:
{{
 "answer": "A/B/C/D",
 "causal_reasoning": "your brief causal explanation"
}}

Question:
{question}

Options:
A. {A}
B. {B}
C. {C}
D. {D}
"""


In [ ]:
def ask_ollama(model, prompt):
    """Call a local Ollama model and return raw text output."""
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return result.stdout.decode("utf-8")


In [ ]:
def extract_json(response):
    try:
        match = re.search(r"\{.*\}", response, re.DOTALL)
        if not match:
            return None
        return json.loads(match.group())
    except:
        return None


In [ ]:
all_results = []

start_time = datetime.now()
print("Started:", start_time)

for model in models:
    print(f"\n============================")
    print(f"Evaluating model: {model}")
    print("============================\n")

    for i, row in enumerate(test_data):
        question = row["question"]
        choices = row["choices"]
        correct_letter = ["A","B","C","D"][row["answer"]]

        prompt = build_prompt(question, choices)
        raw_output = ask_ollama(model, prompt)
        parsed = extract_json(raw_output)

        model_answer = parsed["answer"] if parsed else None
        causal = parsed["causal_reasoning"] if parsed else None

        all_results.append({
            "model": model,
            "question_id": i,
            "question": question,
            "A": choices[0],
            "B": choices[1],
            "C": choices[2],
            "D": choices[3],
            "correct_answer": correct_letter,
            "model_answer": model_answer,
            "is_correct": model_answer == correct_letter,
            "causal_reasoning": causal,
            "raw_output": raw_output
        })

        if i % 10 == 0:
            print(f"{model}: processed {i} questions")

end_time = datetime.now()
print("\nFinished:", end_time)
print("Total time:", end_time - start_time)


Started: 2025-11-25 16:07:25.098406

Evaluating model: llama3.1:8b

llama3.1:8b: processed 0 questions
llama3.1:8b: processed 10 questions
llama3.1:8b: processed 20 questions
llama3.1:8b: processed 30 questions
llama3.1:8b: processed 40 questions
llama3.1:8b: processed 50 questions
llama3.1:8b: processed 60 questions
llama3.1:8b: processed 70 questions
llama3.1:8b: processed 80 questions
llama3.1:8b: processed 90 questions
llama3.1:8b: processed 100 questions
llama3.1:8b: processed 110 questions

Evaluating model: qwen2.5:7b

qwen2.5:7b: processed 0 questions
qwen2.5:7b: processed 10 questions
qwen2.5:7b: processed 20 questions
qwen2.5:7b: processed 30 questions
qwen2.5:7b: processed 40 questions
qwen2.5:7b: processed 50 questions
qwen2.5:7b: processed 60 questions
qwen2.5:7b: processed 70 questions
qwen2.5:7b: processed 80 questions
qwen2.5:7b: processed 90 questions
qwen2.5:7b: processed 100 questions
qwen2.5:7b: processed 110 questions

Evaluating model: qwen2.5:14b

qwen2.5:14b: pr

In [ ]:
df = pd.DataFrame(all_results)
df.head()


,model,question_id,question,A,B,C,D,correct_answer,model_answer,is_correct,causal_reasoning,raw_output
0,llama3.1:8b,0,Which one of the following is the most appropr...,"99% of the time in repeated samples, the inter...","99% of the time in repeated samples, the inter...","99% of the time in repeated samples, the null ...","99% of the time in repeated samples, the null ...",A,A,True,A confidence interval represents a range withi...,"{\n ""answer"": ""A"",\n ""causal_reasoning"": ""A co..."
1,llama3.1:8b,1,What is the main difference between the Dickey...,ADF is a single equation approach to unit root...,PP tests reverse the DF null and alternative h...,The PP test incorporates an automatic correcti...,PP tests have good power in small samples wher...,C,B,False,The Phillips-Perron (PP) approach reverses the...,"{\n ""answer"": ""B"",\n ""causal_reasoning"": ""The ..."
2,llama3.1:8b,2,"If there were a leverage effect in practice, w...",It would rise more quickly for negative distur...,It would be symmetrical about zero,It would rise less quickly for negative distur...,It would be zero for all positive disturbances,A,A,True,"If there were a leverage effect, it means that...","{\n ""answer"": ""A"",\n ""causal_reasoning"": ""If t..."
3,llama3.1:8b,3,Which of the following statements is false con...,There is nothing in the model to ensure that t...,Even if the probabilities are truncated at zer...,The error terms will be heteroscedastic and no...,The model is much harder to estimate than a st...,D,B,False,"In a linear probability model, the estimated p...","{\n ""answer"": ""B"",\n ""causal_reasoning"": ""In a..."
4,llama3.1:8b,4,Which of the following statements concerning t...,The population is the total collection of all ...,The population can be infinite,"In theory, the sample could be larger than the...",A random sample is one where each individual i...,C,D,False,A random sample is not necessarily where each ...,"{\n ""answer"": ""D"",\n ""causal_reasoning"": ""A ra..."


In [ ]:
accuracy = df.groupby("model")["is_correct"].mean()
accuracy


model
llama3.1:8b         0.456140
mistral-nemo:12b    0.456140
qwen2.5:14b         0.666667
qwen2.5:7b          0.631579
Name: is_correct, dtype: float64

In [ ]:
filename = "ollama_mmlu_econometrics_results.csv"
df.to_csv(filename, index=False)
filename


'ollama_mmlu_econometrics_results.csv'